> Experiments with [xsa.util](../trapallada/xsa/util.py)  module

In [1]:
import pathlib
import tempfile
import itertools

import pandas as pd

from astropy.io import fits

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

So that the notebook is aware of modifications in the the `import`ed modules

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## `download_obs` and `extract_filter_tar`

Downloading an observation...

In [3]:
sample_obs = '0404720301'
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(sample_obs, output_directory=tmp_dir_root)

INFO: https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0404720301&level=PPS&instname=PN&instmode=PrimeFullWindow [astroquery.esa.xmm_newton.core]
INFO: Wrote https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0404720301&level=PPS&instname=PN&instmode=PrimeFullWindow to /tmp/tmpn4qcrzri/0404720301.tar [astroquery.esa.xmm_newton.core]


...that is afterwards extracted

In [4]:
sample_obs_dir = trapallada.xsa.util.extract_filter_tar(tar_file)
sample_obs_dir

PosixPath('/tmp/tmpn4qcrzri/0404720301')

Notice that only files whose name matches the regular expression below are kept.

In [5]:
trapallada.xsa.util.re_filename

re.compile(r'^P(\d{10})PN(S\d{3})(SRCTSR)([0-9A-F]+)\.\w+$', re.UNICODE)

Then, e.g.,

In [6]:
trapallada.xsa.util.re_filename.match('P0006010301PNS003SRSPEC0007.FTZ')

is kept but not

In [7]:
trapallada.xsa.util.re_filename.match('XMM_OM_0760_0001_NIXTMS.FIT')

## `download_extract_filter`

Downloading, extracting, filtering using a single function (data used in Javi's paper)

In [8]:
obs = '0605950101' # Javi's paper
obs_dir = trapallada.xsa.util.download_extract_filter(obs, output_directory=tmp_dir_root)

INFO: https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0605950101&level=PPS&instname=PN&instmode=PrimeFullWindow [astroquery.esa.xmm_newton.core]
INFO: Wrote https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0605950101&level=PPS&instname=PN&instmode=PrimeFullWindow to /tmp/tmpn4qcrzri/0605950101.tar [astroquery.esa.xmm_newton.core]


The *first few* downloaded files

In [9]:
list(itertools.islice(obs_dir.glob('*'), 0, 2))

[PosixPath('/tmp/tmpn4qcrzri/0605950101/P0605950101PNS003SRCTSR8036.FTZ'),
 PosixPath('/tmp/tmpn4qcrzri/0605950101/P0605950101PNS003SRCTSR8021.FTZ')]

In [10]:
fits_files = list(obs_dir.glob('*.FTZ'))
fits_files[:3]

[PosixPath('/tmp/tmpn4qcrzri/0605950101/P0605950101PNS003SRCTSR8036.FTZ'),
 PosixPath('/tmp/tmpn4qcrzri/0605950101/P0605950101PNS003SRCTSR8021.FTZ'),
 PosixPath('/tmp/tmpn4qcrzri/0605950101/P0605950101PNS003SRCTSR801A.FTZ')]

A particular file/source is checked

In [11]:
sample_file = obs_dir / 'P0605950101PNS003SRCTSR8001.FTZ'
assert sample_file.exists()

## `read_fits_file`

The file is read into a `DataFrame` (other files, such as e.g. `P0144090201PNX000SRCTSR0000.FTZ`) might contain extra/different columns)

In [12]:
df = trapallada.xsa.util.read_fits_file(sample_file)
df.head(10)

,TIME,RATE,ERROR,FRACEXP,BACKV,BACKE
0,2009-06-03 08:10:33.517454,0.878811,0.878811,0.779384,0.000000,0.000000
1,2009-06-03 08:10:34.977454,0.785685,0.785685,0.871763,0.000000,0.000000
2,2009-06-03 08:10:36.437454,0.000000,0.000000,0.857033,0.000000,0.000000
3,2009-06-03 08:10:37.897454,0.000000,0.000000,0.878588,0.000000,0.000000
4,2009-06-03 08:10:39.357454,0.000000,0.000000,0.874164,0.000000,0.000000
5,2009-06-03 08:10:40.817454,2.033699,1.438042,0.673582,0.000000,0.000000
6,2009-06-03 08:10:42.277454,0.815694,0.815694,0.839691,0.000000,0.000000
7,2009-06-03 08:10:43.737454,0.795970,0.795970,0.860499,0.000000,0.000000
8,2009-06-03 08:10:45.197454,1.563630,1.105653,0.876079,0.000000,0.000000
9,2009-06-03 08:10:46.657454,-0.199282,0.127301,0.857555,0.199303,0.140928


Sampling period can be found as

In [13]:
df['TIME'][1] - df['TIME'][0]

Timedelta('0 days 00:00:01.460000')

## `get_fields_from_header`

One can get fields from the *header* of a file by using

In [14]:
trapallada.xsa.util.get_fields_from_header(sample_file, ['PCOUNT', 'EXPOSURE'])

[0, 39894.570175326]

## `get_position_from_header`

The position being particularly important, there is specific function

In [15]:
trapallada.xsa.util.get_position_from_header(sample_file)

RA      131.755916
DEC     -78.993045
CRA     131.756397
CDEC    -78.993165
dtype: float64

## `bin_fits_df`

Just the `RATE` column is selected for binning

In [16]:
rate_df = trapallada.xsa.util.bin_fits_df(df[['TIME', 'RATE']].set_index('TIME'))
print(f'{len(rate_df)=}')
rate_df

len(rate_df)=504


,RATE
TIME,
2009-06-03 08:12:05.497454,34.577186
2009-06-03 08:13:37.477454,28.529714
2009-06-03 08:15:09.457454,48.805367
2009-06-03 08:16:41.437454,39.296789
2009-06-03 08:18:13.417454,36.757868
...,...
2009-06-03 20:57:03.517454,22.408824
2009-06-03 20:58:35.497454,29.378926
2009-06-03 21:00:07.477454,29.943388


It's plotted

In [17]:
trapallada.plot.df(rate_df, index_type='T')

alt.Chart(...)

## `read_fits_files`

All the above FITs files are read into a `DataFrame`

In [18]:
fits_df, skipped_files_list, header_df = trapallada.xsa.util.read_fits_files(fits_files)
fits_df.head()

,P0605950101PNS003SRCTSR8036.FTZ,P0605950101PNS003SRCTSR8021.FTZ,P0605950101PNS003SRCTSR801A.FTZ,P0605950101PNS003SRCTSR8005.FTZ,P0605950101PNS003SRCTSR802F.FTZ,P0605950101PNS003SRCTSR804C.FTZ,P0605950101PNS003SRCTSR8013.FTZ,P0605950101PNS003SRCTSR800D.FTZ,P0605950101PNS003SRCTSR8037.FTZ,P0605950101PNS003SRCTSR8022.FTZ,...,P0605950101PNS003SRCTSR801C.FTZ,P0605950101PNS003SRCTSR8003.FTZ,P0605950101PNS003SRCTSR8018.FTZ,P0605950101PNS003SRCTSR8030.FTZ,P0605950101PNS003SRCTSR8010.FTZ,P0605950101PNS003SRCTSR8019.FTZ,P0605950101PNS003SRCTSR801B.FTZ,P0605950101PNS003SRCTSR8004.FTZ,P0605950101PNS003SRCTSR800F.FTZ,P0605950101PNS003SRCTSR803C.FTZ
TIME,,,,,,,,,,,,,,,,,,,,,
2009-06-03 08:10:33.517454,0.0,0.0,-0.00934,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.808501,0.000000,-0.02643,0.0,0.0,0.0,0.000000,0.000000,-0.037962
2009-06-03 08:10:34.977454,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.771650,0.000000,0.00000,0.0,0.0,0.0,0.771650,0.000000,0.000000
2009-06-03 08:10:36.437454,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,-0.019985,0.0,...,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,-0.050389,-0.033858
2009-06-03 08:10:37.897454,0.0,0.0,0.00000,0.0,0.0,0.0,-0.043455,-0.017828,0.000000,0.0,...,0.0,0.000000,-0.010292,0.00000,0.0,0.0,0.0,2.329172,0.000000,0.000000
2009-06-03 08:10:39.357454,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.683218,0.000000,0.00000,0.0,0.0,0.0,0.778009,0.000000,0.000000


Some files where a signals couldn't be found were skipped

In [19]:
skipped_files_list[:5]

[]

In [20]:
fits_df.index

DatetimeIndex(['2009-06-03 08:10:33.517454', '2009-06-03 08:10:34.977454',
               '2009-06-03 08:10:36.437454', '2009-06-03 08:10:37.897454',
               '2009-06-03 08:10:39.357454', '2009-06-03 08:10:40.817454',
               '2009-06-03 08:10:42.277454', '2009-06-03 08:10:43.737454',
               '2009-06-03 08:10:45.197454', '2009-06-03 08:10:46.657454',
               ...
               '2009-06-03 21:04:14.217454', '2009-06-03 21:04:15.677454',
               '2009-06-03 21:04:17.137454', '2009-06-03 21:04:18.597454',
               '2009-06-03 21:04:20.057454', '2009-06-03 21:04:21.517454',
               '2009-06-03 21:04:22.977454', '2009-06-03 21:04:24.437454',
               '2009-06-03 21:04:25.897454', '2009-06-03 21:04:27.357454'],
              dtype='datetime64[ns]', name='TIME', length=31805, freq=None)

Information coming from the header of each FIT file (one row per file)

In [21]:
header_df.head()

,RA,DEC,CRA,CDEC
P0605950101PNS003SRCTSR8036.FTZ,132.042026,-79.017632,132.042504,-79.017752
P0605950101PNS003SRCTSR8021.FTZ,131.222406,-78.810052,131.222906,-78.810169
P0605950101PNS003SRCTSR801A.FTZ,130.849340,-78.945602,130.849826,-78.945718
P0605950101PNS003SRCTSR8005.FTZ,131.068342,-78.985571,131.068823,-78.985688
P0605950101PNS003SRCTSR802F.FTZ,131.425536,-79.119158,131.426003,-79.119277


Binning is performed

In [22]:
binned_df = trapallada.xsa.util.bin_fits_df(fits_df, bin_size=63)

In [23]:
trapallada.plot.df(binned_df, columns=[pathlib.Path(c).stem for c in fits_df.columns], index_type='T')

alt.Chart(...)

In order to get the *most promising* (meaningful) sources from some observation, its associated (*untar*ed) path is explored

In [24]:
n_sources_per_obs = 3
ranked_files_df = trapallada.xsa.util.rank_obs_files(obs_dir).iloc[:n_sources_per_obs]['full path']
ranked_files_df

37    /tmp/tmpn4qcrzri/0605950101/P0605950101PNS003S...
29    /tmp/tmpn4qcrzri/0605950101/P0605950101PNS003S...
40    /tmp/tmpn4qcrzri/0605950101/P0605950101PNS003S...
Name: full path, dtype: object

High-level function to read a *tar* file into a `DataFrame`

In [25]:
tar_file_df, tar_file_headers_df = trapallada.xsa.util.read_tar(tar_file)
tar_file_df

,P0404720301PNS003SRCTSR8001.FTZ,P0404720301PNS003SRCTSR8002.FTZ,P0404720301PNS003SRCTSR8003.FTZ
TIME,,,
2007-01-15 18:55:51.589320,-0.101347,-0.138722,-0.086892
2007-01-15 18:55:53.049320,-0.179067,-0.265774,-0.083237
2007-01-15 18:55:54.509320,-0.185731,1.307495,-0.108944
2007-01-15 18:55:55.969320,-0.194821,1.188581,-0.110012
2007-01-15 18:55:57.429320,-0.292804,0.000000,1.429809
...,...,...,...
2007-01-16 04:39:42.829320,0.000000,0.000000,0.000000
2007-01-16 04:39:44.289320,0.000000,0.000000,0.000000
2007-01-16 04:39:45.749320,0.000000,0.000000,0.000000


In [26]:
tar_file_headers_df

,RA,DEC,CRA,CDEC
P0404720301PNS003SRCTSR8001.FTZ,351.257658,61.563009,351.257885,61.563287
P0404720301PNS003SRCTSR8002.FTZ,351.602702,61.718042,351.602615,61.718478
P0404720301PNS003SRCTSR8003.FTZ,351.500044,61.723238,351.499946,61.723627


Slicing

In [27]:
window_size = 200
step = 100
slices, time_slices = trapallada.xsa.util.slice(tar_file_df, window_size=window_size, step=step)
slices.shape, time_slices.shape

((717, 200), (717, 200))

First element is the 1st window for the 1st signal, second element is the 1st window for the 2nd signal, and so forth and so on. Hence,

In [28]:
(time_slices[0] == time_slices[1]).all()

True

A `DataFrame` with **all** the timestamps

In [29]:
time_df = pd.DataFrame(time_slices)
time_df.head()

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
0,2007-01-15 18:55:51.589320,2007-01-15 18:55:53.049320,2007-01-15 18:55:54.509320,2007-01-15 18:55:55.969320,2007-01-15 18:55:57.429320,2007-01-15 18:55:58.889320,2007-01-15 18:56:00.349320,2007-01-15 18:56:01.809320,2007-01-15 18:56:03.269320,2007-01-15 18:56:04.729320,...,2007-01-15 19:00:28.989320,2007-01-15 19:00:30.449320,2007-01-15 19:00:31.909320,2007-01-15 19:00:33.369320,2007-01-15 19:00:34.829320,2007-01-15 19:00:36.289320,2007-01-15 19:00:37.749320,2007-01-15 19:00:39.209320,2007-01-15 19:00:40.669320,2007-01-15 19:00:42.129320
1,2007-01-15 18:55:51.589320,2007-01-15 18:55:53.049320,2007-01-15 18:55:54.509320,2007-01-15 18:55:55.969320,2007-01-15 18:55:57.429320,2007-01-15 18:55:58.889320,2007-01-15 18:56:00.349320,2007-01-15 18:56:01.809320,2007-01-15 18:56:03.269320,2007-01-15 18:56:04.729320,...,2007-01-15 19:00:28.989320,2007-01-15 19:00:30.449320,2007-01-15 19:00:31.909320,2007-01-15 19:00:33.369320,2007-01-15 19:00:34.829320,2007-01-15 19:00:36.289320,2007-01-15 19:00:37.749320,2007-01-15 19:00:39.209320,2007-01-15 19:00:40.669320,2007-01-15 19:00:42.129320
2,2007-01-15 18:55:51.589320,2007-01-15 18:55:53.049320,2007-01-15 18:55:54.509320,2007-01-15 18:55:55.969320,2007-01-15 18:55:57.429320,2007-01-15 18:55:58.889320,2007-01-15 18:56:00.349320,2007-01-15 18:56:01.809320,2007-01-15 18:56:03.269320,2007-01-15 18:56:04.729320,...,2007-01-15 19:00:28.989320,2007-01-15 19:00:30.449320,2007-01-15 19:00:31.909320,2007-01-15 19:00:33.369320,2007-01-15 19:00:34.829320,2007-01-15 19:00:36.289320,2007-01-15 19:00:37.749320,2007-01-15 19:00:39.209320,2007-01-15 19:00:40.669320,2007-01-15 19:00:42.129320
3,2007-01-15 18:58:17.589320,2007-01-15 18:58:19.049320,2007-01-15 18:58:20.509320,2007-01-15 18:58:21.969320,2007-01-15 18:58:23.429320,2007-01-15 18:58:24.889320,2007-01-15 18:58:26.349320,2007-01-15 18:58:27.809320,2007-01-15 18:58:29.269320,2007-01-15 18:58:30.729320,...,2007-01-15 19:02:54.989320,2007-01-15 19:02:56.449320,2007-01-15 19:02:57.909320,2007-01-15 19:02:59.369320,2007-01-15 19:03:00.829320,2007-01-15 19:03:02.289320,2007-01-15 19:03:03.749320,2007-01-15 19:03:05.209320,2007-01-15 19:03:06.669320,2007-01-15 19:03:08.129320
4,2007-01-15 18:58:17.589320,2007-01-15 18:58:19.049320,2007-01-15 18:58:20.509320,2007-01-15 18:58:21.969320,2007-01-15 18:58:23.429320,2007-01-15 18:58:24.889320,2007-01-15 18:58:26.349320,2007-01-15 18:58:27.809320,2007-01-15 18:58:29.269320,2007-01-15 18:58:30.729320,...,2007-01-15 19:02:54.989320,2007-01-15 19:02:56.449320,2007-01-15 19:02:57.909320,2007-01-15 19:02:59.369320,2007-01-15 19:03:00.829320,2007-01-15 19:03:02.289320,2007-01-15 19:03:03.749320,2007-01-15 19:03:05.209320,2007-01-15 19:03:06.669320,2007-01-15 19:03:08.129320


The durations can be easily obtained...

In [30]:
durations = time_df.iloc[:, -1] - time_df.iloc[:, 0]
durations.head()

0   0 days 00:04:50.540000
1   0 days 00:04:50.540000
2   0 days 00:04:50.540000
3   0 days 00:04:50.540000
4   0 days 00:04:50.540000
dtype: timedelta64[ns]

...as well as the periods

In [31]:
periods = time_df.iloc[:, 1] - time_df.iloc[:, 0]
periods.head()

0   0 days 00:00:01.460000
1   0 days 00:00:01.460000
2   0 days 00:00:01.460000
3   0 days 00:00:01.460000
4   0 days 00:00:01.460000
dtype: timedelta64[ns]

## `tars_to_array`

The *tar* files downloaded above (into a temporary directory)...

In [32]:
list(tmp_dir_root.glob('*.tar'))

[PosixPath('/tmp/tmpn4qcrzri/0404720301.tar'),
 PosixPath('/tmp/tmpn4qcrzri/0605950101.tar')]

...are turned into an `Array` (and a companion `DataFrame` with metadata)

In [33]:
signals, metadata_df, headers_df  = trapallada.xsa.util.tars_to_array(tmp_dir_root.glob('*.tar'))
signals.shape, metadata_df.shape, headers_df.shape

trapallada.xsa.util - INFO - Requested a window size of, effectively, duration 18396.0 seconds (period is 91.98 seconds, step is 4599.0 seconds)


((52, 200), (52, 4), (8, 4))

Metadata for the headers (notice that every FIT file is *sliced* into multiple signals)

In [34]:
headers_df

RA        DEC  \
tar file       source                                                   
0404720301.tar P0404720301PNS003SRCTSR8001.FTZ  351.257658  61.563009   
               P0404720301PNS003SRCTSR8002.FTZ  351.602702  61.718042   
               P0404720301PNS003SRCTSR8003.FTZ  351.500044  61.723238   
               P0404720301PNS003SRCTSR8004.FTZ  351.227955  61.558033   
0605950101.tar P0605950101PNS003SRCTSR8001.FTZ  131.755916 -78.993045   
               P0605950101PNS003SRCTSR8002.FTZ  130.779291 -79.081070   
               P0605950101PNS003SRCTSR8003.FTZ  130.598155 -79.067407   
               P0605950101PNS003SRCTSR8004.FTZ  130.799788 -79.070072   

                                                       CRA       CDEC  
tar file       source                                                  
0404720301.tar P0404720301PNS003SRCTSR8001.FTZ  351.257885  61.563287  
               P0404720301PNS003SRCTSR8002.FTZ  351.602615  61.718478  
               P0404720301PNS003SRCTSR8003.FTZ  351.499946  61.723627  
               P0404720301PNS003SRCTSR8004.FTZ  351.228191  61.558297  
0605950101.tar P0605950101PNS003SRCTSR8001.FTZ  131.756397 -78.993165  
               P0605950101PNS003SRCTSR8002.FTZ  130.779762 -79.081186  
               P0605950101PNS003SRCTSR8003.FTZ  130.598627 -79.067521  
               P0605950101PNS003SRCTSR8004.FTZ  130.800260 -79.070188

Durations:

In [35]:
(metadata_df['end'] - metadata_df['start']).head()

0   0 days 05:05:04.020000
1   0 days 05:05:04.020000
2   0 days 05:05:04.020000
3   0 days 05:05:04.020000
4   0 days 05:05:04.020000
dtype: timedelta64[ns]

For one particular relevant observation

In [36]:
relevant_source = metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ'
metadata_df[relevant_source]

,start,end,tar file,source
20,2009-06-03 08:12:05.497454,2009-06-03 13:17:09.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
24,2009-06-03 09:28:44.497454,2009-06-03 14:33:48.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
28,2009-06-03 10:45:23.497454,2009-06-03 15:50:27.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
32,2009-06-03 12:02:02.497454,2009-06-03 17:07:06.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
36,2009-06-03 13:18:41.497454,2009-06-03 18:23:45.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
40,2009-06-03 14:35:20.497454,2009-06-03 19:40:24.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
44,2009-06-03 15:51:59.497454,2009-06-03 20:57:03.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
48,2009-06-03 15:58:07.417454,2009-06-03 21:03:11.437454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ


The data from the header

In [37]:
headers_df.xs('P0605950101PNS003SRCTSR8001.FTZ', level='source')

,RA,DEC,CRA,CDEC
tar file,,,,
0605950101.tar,131.755916,-78.993045,131.756397,-78.993165


In [38]:
trapallada.plot.signals_array(signals[relevant_source][-1:])

alt.Chart(...)